In [1]:
import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/Abhijeet/Documents/coding-repos/neural-networks/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/Abhijeet/Documents/coding-repos/neural-networks/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/Abhijeet/Documents/coding-repos/neural-networks/.venv/lib

In [2]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
print(housing.data.shape, housing.target.shape)
print(housing.feature_names)
# print(housing.DESCR)

(20640, 8) (20640,)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [3]:
x = housing.data
y = housing.target
print(len(x), len(y))

20640 20640


In [4]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((16512, 8), (16512,), (4128, 8), (4128,))

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

print(x_train.mean(axis=0).round(2))
print(x_train.std(axis=0).round(2))

[-0. -0.  0. -0. -0. -0.  0. -0.]
[1. 1. 1. 1. 1. 1. 1. 1.]


In [6]:
x_test = torch.tensor(x_test, dtype=torch.float32)
x_train = torch.tensor(x_train, dtype=torch.float32)

y_test = torch.tensor(y_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

x_train[0].shape, x_test.dtype

(torch.Size([8]), torch.float32)

In [7]:
print(y_train.shape, y_test.shape)

y_train = y_train.view(-1,1)
y_test = y_test.view(-1,1)

y_train.shape, y_test.shape

torch.Size([16512]) torch.Size([4128])


(torch.Size([16512, 1]), torch.Size([4128, 1]))

In [8]:
dataset = TensorDataset(x_train, y_train)
print(len(dataset))
sample_features, sample_label = dataset[0]
sample_features, sample_label

16512


(tensor([-0.3262,  0.3485, -0.1749, -0.2084,  0.7683,  0.0514, -1.3728,  1.2726]),
 tensor([1.0300]))

In [9]:
loader = DataLoader(dataset, batch_size=16, shuffle=True)
len(loader)

1032

In [10]:
torch.manual_seed(0)

W1 = (torch.randn(8, 32) * (2 / 8) ** 0.5).requires_grad_()
b1 = torch.zeros(32, requires_grad=True)
W2 = (torch.randn(32, 64) * (2 / 32) ** 0.5).requires_grad_()
b2 = torch.zeros(64, requires_grad=True)
W3 = (torch.randn(64, 1) * (2 / 64) ** 0.5).requires_grad_()
b3 = torch.zeros(1, requires_grad=True)

params = [W1, b1, W2, b2, W3, b3]

In [11]:
def forward(x):
    h1 = torch.clamp(x @ W1 + b1, min=0)
    h2 = torch.clamp(h1 @ W2 + b2, min=0)
    return h2 @ W3 + b3

In [12]:
lr = 0.01
batch_size = 64
n = x_train.shape[0]

for epoch in range(50):
    perm = torch.randperm(n)
    total_loss = 0
    num_batches = 0

    for i in range(0, n, batch_size):
        idx = perm[i:i + batch_size]
        x_batch, y_batch = x_train[idx], y_train[idx]

        preds = forward(x_batch)
        loss = ((preds - y_batch) ** 2).mean()

        loss.backward()

        with torch.no_grad():
            for p in params:
                p -= lr * p.grad
                p.grad.zero_()

        total_loss += loss.item()
        num_batches += 1

    print(f"Epoch {epoch + 1}: loss = {total_loss / num_batches:.4f}")

Epoch 1: loss = 1.6332
Epoch 2: loss = 1.7441
Epoch 3: loss = 0.6088
Epoch 4: loss = 0.4710
Epoch 5: loss = 0.4389
Epoch 6: loss = 0.4211
Epoch 7: loss = 0.4081
Epoch 8: loss = 0.3968
Epoch 9: loss = 0.3871
Epoch 10: loss = 0.3795
Epoch 11: loss = 0.3726
Epoch 12: loss = 0.3671
Epoch 13: loss = 0.3625
Epoch 14: loss = 0.3578
Epoch 15: loss = 0.3533
Epoch 16: loss = 0.3496
Epoch 17: loss = 0.3467
Epoch 18: loss = 0.3425
Epoch 19: loss = 0.3410
Epoch 20: loss = 0.3372
Epoch 21: loss = 0.3348
Epoch 22: loss = 0.3323
Epoch 23: loss = 0.3299
Epoch 24: loss = 0.3275
Epoch 25: loss = 0.3262
Epoch 26: loss = 0.3240
Epoch 27: loss = 0.3231
Epoch 28: loss = 0.3207
Epoch 29: loss = 0.3188
Epoch 30: loss = 0.3171
Epoch 31: loss = 0.3153
Epoch 32: loss = 0.3145
Epoch 33: loss = 0.3132
Epoch 34: loss = 0.3117
Epoch 35: loss = 0.3106
Epoch 36: loss = 0.3098
Epoch 37: loss = 0.3082
Epoch 38: loss = 0.3060
Epoch 39: loss = 0.3054
Epoch 40: loss = 0.3050
Epoch 41: loss = 0.3029
Epoch 42: loss = 0.3025
E

In [14]:
with torch.no_grad():
    test_preds = forward(x_test)
    test_mse = ((test_preds - y_test) ** 2).mean()
    test_rmse = test_mse.sqrt() * 100000

    baseline = y_train.mean()
    base_mse = ((baseline - y_test) ** 2).mean()
    base_rmse = base_mse.sqrt() * 100000

print(f"Model test error:    ${test_rmse:,.0f}")
print(f"Baseline test error: ${base_rmse:,.0f}")
print(test_preds[:5].view(-1))
print(y_test[:5].view(-1))

Model test error:    $56,109
Baseline test error: $114,486
tensor([0.5633, 1.3084, 4.0786, 2.6504, 2.8799])
tensor([0.4770, 0.4580, 5.0000, 2.1860, 2.7800])
